# Covector Lie Neurons — Equivariance 검증 노트북

**목적.** Point cloud 입력에 SE(3) transformation $T=(R,p)$를 가했을 때, 네트워크 출력이
원하는 transformation rule을 따르는지 수치적으로 확인한다.

| 대상 | 검증할 법칙 |
|---|---|
| Wrench encoder $W(P)$ | $W(T\cdot P) = \mathrm{Ad}_T^{-\top}\, W(P)$ |
| 출력 $L$ ($K=LL^\top$의 factor) | $L(T\cdot P) = \mathrm{Ad}_T^{-\top}\, L(P)$ |
| Stiffness $K$ | $K(T\cdot P) = \mathrm{Ad}_T^{-\top}\, K(P)\, \mathrm{Ad}_T^{-1}$ (congruence) |

파이프라인 (전 구간 covector-native, $Q$ 없음):

```
P ──(pure-force lift: W(r,n) = (r×n, n))──▶ R^{6×C}
  ──(LNLinear + covector bracket [·,·]*)──▶ Z
  ──(L = Z/√C,  K = LLᵀ)──▶ 출력
```

모든 가중치는 **랜덤 초기화** (학습 없음) — equivariance는 구조적 성질이므로 임의의
가중치에서 성립해야 한다. float64로 계산하여 round-off ($\sim 10^{-16}$)와 구조적
실패 ($O(1)$)를 명확히 구분한다.

> ⚠️ 판정은 반드시 $p \neq 0$에서 해야 한다. 모든 타입 오류는 $p=0$ (회전만)에서는
> 보이지 않는다 ($\mathrm{Ad}_{(R,0)}$이 직교라 $\mathrm{Ad} = \mathrm{Ad}^{-\top}$).


In [9]:
import os
import sys

# repo 루트를 sys.path에 추가 (core/, experiment/ 를 임포트하기 위함)
d = os.getcwd()
while not os.path.isdir(os.path.join(d, 'core')):
    parent = os.path.dirname(d)
    assert parent != d, 'repo root not found — run from inside the repository'
    d = parent
sys.path.insert(0, d)

import torch

torch.set_default_dtype(torch.float64)   # round-off vs 구조적 실패를 분리
SEED = 0

from experiment.pc_se3_congruence.se3_utils import (
    random_SE3, coadjoint, transform_cloud, scaled_err)
from experiment.pc_se3_congruence.encoders import (
    WrenchPlueckerEncoder, WrenchLearnableLiftEncoder)
from experiment.pc_se3_congruence.models import (
    ModelPC2K, ModelPC2KNaiveBracket)

print('torch', torch.__version__, '| default dtype:', torch.get_default_dtype())

torch 2.11.0+cu128 | default dtype: torch.float64


## 0. 공통 셋업

- 랜덤 point cloud $P \in \mathbb{R}^{B \times N \times 3}$
- translation 크기 $\lVert p \rVert \sim \{0,\ 1,\ 10^2,\ 10^4\}$ 를 sweep, scale마다 5회 시도 후 최대 오차 보고
- 오차 metric: scale-free relative error $\;e = \dfrac{\lVert A - B\rVert_F}{\max(\lVert A\rVert_F, \lVert B\rVert_F)}$


In [10]:
TRANS_SCALES = [0.0, 1.0, 1e2, 1e4]
N_TRIALS = 5

gen = torch.Generator().manual_seed(SEED)
P = torch.randn(2, 64, 3, generator=gen)          # [B, N, 3]

def coad_apply(A, W):
    '''wrench feature [B, C, 6, N] 에 6x6 행렬 A를 geometric 차원에 적용'''
    return torch.einsum('ij,bcjn->bcin', A, W)

def sweep(err_fn):
    '''scale별로 N_TRIALS회 T를 샘플링해 err_fn(R, p)의 최대값을 모은다.'''
    out = []
    for s in TRANS_SCALES:
        errs = [err_fn(*random_SE3(s, gen)) for _ in range(N_TRIALS)]
        out.append(max(errs))
    return out

def report(rows, title):
    w = max(len(name) for name, _ in rows) + 2
    head = 'p=0        |p|~1      |p|~1e2    |p|~1e4'
    print(f'\n{title}\n' + '-' * (w + len(head)))
    print(' ' * w + head)
    for name, errs in rows:
        print(name.ljust(w) + '  '.join(f'{e:.1e}' for e in errs))

---
# 네트워크 구현 상세

검증에 들어가기 전에, 파이프라인의 각 단계가 **무엇을 어떻게 계산하고 왜 equivariant한지**
정리한다. 전체 데이터 흐름과 shape:

$$\underbrace{P}_{[B,N,3]}
\;\xrightarrow{\text{encoder}}\;
\underbrace{W}_{[B,C_0,6,1]}
\;\xrightarrow{\text{backbone}}\;
\underbrace{Z}_{[B,C,6,1]}
\;\xrightarrow{\text{head}}\;
\underbrace{L}_{[B,6,C]},\;
\underbrace{K}_{[B,6,6]}$$

feature tensor의 축은 `[batch, 채널 C, geometric 6, 원소 N]`이다. **geometric 차원(6)**이
wrench 좌표이고, 여기에만 group action이 걸린다. 저장 순서는 `[f; m]`
(slot 0–2 = force $f$, slot 3–5 = moment $m$) — twist의 `[v; ω]`와 거울상이다.
group action은 왼쪽 곱 $W \mapsto \mathrm{Ad}_T^{-\top} W$ (6×6 행렬을 geometric 축에 적용),
학습 가중치는 전부 **오른쪽 곱**(채널 축)이라 둘이 commute하는 것이 equivariance의 핵심 원리다.

## A. Encoder — point cloud를 wrench feature로

두 인코더 모두 같은 수학적 연산(pure-force screw lifting)의 다른 파라미터화다:

$$W(r, n) = (m, f) = (r \times n,\; n)
\qquad \text{("점 } r \text{을 지나 방향 } n \text{으로 작용하는 단위 힘의 origin 기준 wrench")}$$

**(1) `WrenchPlueckerEncoder` — closed form, 파라미터 0개.**

1. kNN 그래프: $d(r_i, r_j)$는 SE(3)-invariant → 이웃 집합과 거리 순위도 invariant
2. 방향장: $n_{ij} = r_j - r_i$ (pairwise difference — translation에서 $p$가 소거되고 회전만 남음: $n' = Rn$)
3. lift: $f_{ij} = n_{ij}$, $m_{ij} = r_i \times n_{ij}$
4. 채널 구성: 채널 $c$ = "거리 순위 $c$번째 이웃" (순위가 invariant하므로 채널 정렬이 $T$에 안 흔들림)
5. 점들에 대해 mean → $W \in [B, k, 6, 1]$

**(2) `WrenchLearnableLiftEncoder` — 학습형 방향장.**

방향장 $n$을 pairwise difference 대신 작은 VN-DGCNN이 출력한다:

- 입력을 centroid $c$로 centering: $x_i = r_i - c$ (translation invariance 확보; $c$는 equivariant하게 움직이므로 $x_i$는 회전만 받음)
- `VNEdgeConv` ×2 → `VNLinear` → `VNLeakyReLU` ($1 \to 8 \to 8 \to C$ 채널). 모든 층이
  3-벡터를 3-벡터로 보내는 SO(3)-equivariant 연산 (Vector Neurons; 내적/norm 기반 folding은
  SO(3)-invariant라 안전) → $n_i^{(ch)}$은 rotation-only: $n' = Rn$
- anchor에서 lift: $f = n$, $m = (r - c) \times n$
- **anchor transport**: moment가 데이터 의존적인 anchor $c$ 기준이므로 coadjoint
  $\mathrm{Ad}_{(I,c)}^{-\top}: (m, f) \mapsto (m + c \times f,\; f)$ 로 origin 기준으로 되돌린다.
  이걸 생략하면 $SO(3)\times SO(3)$로 붕괴 (negative control 2에서 확인)

두 경우 모두 방향장에 필요한 조건은 단 하나 — **translation-invariant + SO(3)-equivariant**
($n' = Rn$). 그러면 §2.2의 한 줄 증명에 의해 $W(T\cdot P) = \mathrm{Ad}_T^{-\top} W(P)$.

## B. Backbone — `CovectorBackbone` (LNLinear + covector bracket)

`channels = (8, 16, 16, 8)`: 블록 3개, 각 블록은 `LNLinearAndCovectorBracket` =

**B-1. `LNLinear`** (Lie Neurons 논문 식 8 그대로, 무수정 재사용):

$$x' = x\,W_{\text{lin}}, \qquad W_{\text{lin}} \in \mathbb{R}^{C \times C'},\ \text{bias 없음}$$

채널 축(오른쪽) 곱이므로 geometric 축(왼쪽)에 걸리는 $\mathrm{Ad}_T^{-\top}$와 자동으로
commute — **어떤** 왼쪽 표현에 대해서도 equivariant라 covector에서도 무수정으로 성립.
bias는 addition이 왼쪽 action과 commute하지 않아 금지 (negative control A4).

**B-2. `LNCovectorBracket`** (논문 식 12의 LN-Bracket에서 bracket만 교체):

$$x' = x + [\,xU,\; xV\,]_* \qquad U, V \in \mathbb{R}^{C \times C} \text{ (학습, bias 없음)}$$

- $d_1 = xU$, $d_2 = xV$: 두 개의 학습된 equivariant wrench feature (LNLinear과 같은 원리)
- covector bracket은 closed form — $Q$ 행렬 없이 cross product 4번:

$$[F_1, F_2]_* = \bigl(\underbrace{f_1 \times m_2 - f_2 \times m_1}_{\text{moment 슬롯}},\;
\underbrace{f_1 \times f_2}_{\text{force 슬롯}}\bigr)$$

- skip connection $x + (\cdot)$: $[X,X]=0$이라 bracket 출력이 실전에서 작을 수 있어
  정보 흐름 보강용 (논문과 동일한 설계 이유). 두 항 모두 equivariant하므로 합도 equivariant.
- **비선형성은 이 bracket이 유일하다.** Killing form 기반 층(LN-ReLU/BatchNorm/MaxPool)은
  $\mathfrak{se}(3)$가 degenerate해서 쓰지 않는다 (moment ideal에 blind → 게이트가 identity로 붕괴).

**B-3. Pooling이 없는 이유**: 인코더가 이미 점들에 대해 mean pooling을 마쳤으므로
백본에는 원소 축 $N=1$만 남아 있다. $N>1$ 구조로 바꾸면 mean pooling(선형이라 equivariant)을 쓰면 된다.

## C. Head — `CovectorGramHeadLK`

$$L = Z / \sqrt{C} \in \mathbb{R}^{6 \times C}, \qquad K = L L^\top \in \mathbb{R}^{6 \times 6}$$

- $Z \mapsto \mathrm{Ad}_T^{-\top} Z$이므로 $L \mapsto \mathrm{Ad}_T^{-\top} L$이 **정의상** 성립
  — 원했던 법칙 $f(T\cdot PC) = \mathrm{Ad}_T^{-\top} f(PC)$가 head에서 공짜로 나온다
- $K = LL^\top \mapsto \mathrm{Ad}_T^{-\top} K \mathrm{Ad}_T^{-1}$ (congruence), symmetric PSD by construction,
  $C \ge 6$이면 generically rank 6 (SPD)
- ⚠️ 순서 주의: **$L$을 출력하고 $K$를 조립**한다. 반대로 $K$에서 Cholesky로 $L$을 뽑으면
  equivariance가 깨진다 (Cholesky는 기저 순서 의존)

## D. Equivariance 원리 한 장 요약

| 단계 | 연산 | equivariant한 이유 |
|---|---|---|
| encoder | $W(r,n)=(r\times n, n)$ | 한 줄 증명: $m' = Rm + p\times Rf$, $f' = Rf$ |
| LNLinear | $xW_{\text{lin}}$ (오른쪽 곱) | 왼쪽 action과 commute (표현 무관) |
| covector bracket | $x + [xU, xV]_*$ | $[\cdot,\cdot]_*$가 coadjoint의 (유일한) equivariant bilinear 연산 |
| head | $L = Z/\sqrt{C}$, $K=LL^\top$ | 스칼라 배 + congruence 조립 |

모든 학습 파라미터($W_{\text{lin}}, U, V$)는 채널 축에만 작용 → **equivariance는 가중치 값과
무관한 구조적 성질**. 그래서 아래 검증은 전부 랜덤 초기화로 수행한다.

---

## 1. Encoder equivariance — $W(T\cdot P) = \mathrm{Ad}_T^{-\top} W(P)$

pure-force wrench lifting $W(r,n) = (m, f) = (r\times n,\; n)$:

- **WrenchPlueckerEncoder**: $n$ = kNN pairwise difference (closed form, 파라미터 없음)
- **WrenchLearnableLiftEncoder**: $n$ = VN-DGCNN direction field (랜덤 가중치)


In [11]:
torch.manual_seed(SEED)
enc_plueck = WrenchPlueckerEncoder(k=8)
enc_learn = WrenchLearnableLiftEncoder(out_channels=8, k=8, mode='anchor_transport')

def enc_err(enc):
    def fn(R, p):
        TP = transform_cloud(P, R, p)
        return scaled_err(enc(TP), coad_apply(coadjoint(R, p), enc(P)))
    return fn

rows_enc = [
    ('Pluecker (closed form)', sweep(enc_err(enc_plueck))),
    ('Learnable (VN-DGCNN)  ', sweep(enc_err(enc_learn))),
]
report(rows_enc, '[1] Encoder:  W(T.P)  vs  Ad^{-T} W(P)')


[1] Encoder:  W(T.P)  vs  Ad^{-T} W(P)
----------------------------------------------------------------
                        p=0        |p|~1      |p|~1e2    |p|~1e4
Pluecker (closed form)  3.7e-16  3.7e-16  6.3e-15  7.0e-13
Learnable (VN-DGCNN)    2.7e-15  1.7e-15  1.3e-13  1.1e-11


## 2. End-to-end — $L$과 $K$의 transformation rule

`ModelPC2K` = wrench encoder → `CovectorBackbone` (LNLinear + covector bracket) → $(L,\ K=LL^\top)$

- $L(T\cdot P) \overset{?}{=} \mathrm{Ad}_T^{-\top} L(P)$  — 요구했던 법칙 그대로
- $K(T\cdot P) \overset{?}{=} \mathrm{Ad}_T^{-\top} K(P) \mathrm{Ad}_T^{-1}$ — congruence


In [12]:
torch.manual_seed(SEED)
model_p = ModelPC2K(WrenchPlueckerEncoder(k=8)).eval()
model_l = ModelPC2K(WrenchLearnableLiftEncoder(out_channels=8, k=8)).eval()

# ---- 실제 모듈 구조와 파라미터 수 ----
def n_params(m):
    return sum(p.numel() for p in m.parameters())

print(model_p)
print(f'\n파라미터 수: encoder={n_params(model_p.encoder)}, '
      f'backbone={n_params(model_p.backbone)}, head={n_params(model_p.head)}')
print(f'(learnable encoder 버전의 encoder 파라미터 수: {n_params(model_l.encoder)})')

# shape 흐름 확인
with torch.no_grad():
    W0 = model_p.encoder(P)
    Z0 = model_p.backbone(W0)
    L0, K0 = model_p.head(Z0)
print(f'\nshape:  P {tuple(P.shape)}  ->  W {tuple(W0.shape)}'
      f'  ->  Z {tuple(Z0.shape)}  ->  L {tuple(L0.shape)},  K {tuple(K0.shape)}')

def e2e_errs(model):
    def fn_L(R, p):
        with torch.no_grad():
            L, _ = model(P)
            LT, _ = model(transform_cloud(P, R, p))
        return scaled_err(LT, coadjoint(R, p) @ L)

    def fn_K(R, p):
        A = coadjoint(R, p)
        with torch.no_grad():
            _, K = model(P)
            _, KT = model(transform_cloud(P, R, p))
        return scaled_err(KT, A @ K @ A.transpose(-1, -2))
    return fn_L, fn_K

fL_p, fK_p = e2e_errs(model_p)
fL_l, fK_l = e2e_errs(model_l)
rows_e2e = [
    ('L  (Pluecker enc.) ', sweep(fL_p)),
    ('K  (Pluecker enc.) ', sweep(fK_p)),
    ('L  (learnable enc.)', sweep(fL_l)),
    ('K  (learnable enc.)', sweep(fK_l)),
]
report(rows_e2e, '[2] End-to-end:  L -> Ad^{-T} L,   K -> Ad^{-T} K Ad^{-1}')

ModelPC2K(
  (encoder): WrenchPlueckerEncoder()
  (backbone): CovectorBackbone(
    (blocks): ModuleList(
      (0): LNLinearAndCovectorBracket(
        (linear): LNLinear(
          (fc): Linear(in_features=8, out_features=16, bias=False)
        )
        (bracket): LNCovectorBracket(
          (learn_dir): Linear(in_features=16, out_features=16, bias=False)
          (learn_dir2): Linear(in_features=16, out_features=16, bias=False)
        )
      )
      (1): LNLinearAndCovectorBracket(
        (linear): LNLinear(
          (fc): Linear(in_features=16, out_features=16, bias=False)
        )
        (bracket): LNCovectorBracket(
          (learn_dir): Linear(in_features=16, out_features=16, bias=False)
          (learn_dir2): Linear(in_features=16, out_features=16, bias=False)
        )
      )
      (2): LNLinearAndCovectorBracket(
        (linear): LNLinear(
          (fc): Linear(in_features=16, out_features=8, bias=False)
        )
        (bracket): LNCovectorBracket(
         

## 3. $K$의 성질 — symmetric PSD, $C \ge 6$이면 generically SPD


In [13]:
with torch.no_grad():
    _, K = model_p(P)
sym_err = scaled_err(K, K.transpose(-1, -2))
eigs = torch.linalg.eigvalsh(K)
print(f'symmetry error   : {sym_err:.1e}')
print(f'min eigenvalue   : {eigs.min().item():.3e}  (>= 0 이어야 함)')
print(f'rank (per batch) : {[int(torch.linalg.matrix_rank(K[b]).item()) for b in range(K.shape[0])]}')

symmetry error   : 0.0e+00
min eigenvalue   : 3.925e-07  (>= 0 이어야 함)
rank (per batch) : [6, 6]


## 4. Cascade (합성 법칙)

$T_2 T_1$을 한 번에 가한 결과가, $T_1$의 출력에 $\mathrm{Ad}_{T_2}^{-\top}(\cdot)\mathrm{Ad}_{T_2}^{-1}$을
가한 것과 같아야 한다 (compliance control에서 프레임을 연쇄적으로 옮길 때 필요한 성질):

$$K(T_2 T_1 \cdot P) \overset{?}{=} \mathrm{Ad}_{T_2}^{-\top}\, K(T_1 \cdot P)\, \mathrm{Ad}_{T_2}^{-1}$$


In [14]:
from experiment.pc_se3_congruence.se3_utils import compose

def cascade_err(R2, p2):
    R1, p1 = random_SE3(1.0, gen)
    R21, p21 = compose((R2, p2), (R1, p1))
    A2 = coadjoint(R2, p2)
    with torch.no_grad():
        _, K1 = model_p(transform_cloud(P, R1, p1))
        _, K21 = model_p(transform_cloud(P, R21, p21))
    return scaled_err(K21, A2 @ K1 @ A2.transpose(-1, -2))

report([('cascade (Pluecker)', sweep(cascade_err))],
       '[4] Cascade:  K(T2 T1 . P)  vs  Ad_{T2}^{-T} K(T1 . P) Ad_{T2}^{-1}')


[4] Cascade:  K(T2 T1 . P)  vs  Ad_{T2}^{-T} K(T1 . P) Ad_{T2}^{-1}
------------------------------------------------------------
                    p=0        |p|~1      |p|~1e2    |p|~1e4
cascade (Pluecker)  1.0e-15  1.0e-15  1.0e-14  6.8e-13


## 5. Negative controls — 테스트가 실제로 위반을 잡아내는가

각 control은 가정 하나씩만 깨뜨린다. **$p=0$에서는 전부 통과**하고 $p \neq 0$에서만
$O(1)$로 실패해야 정상이다 (= 이 테스트 스위트가 변별력이 있다는 증거).

1. **twist bracket을 wrench에 그대로 적용** (`ModelPC2KNaiveBracket`) — bracket은
   $\mathrm{Ad}$-equivariant이지 $\mathrm{Ad}^{-\top}$-equivariant가 아님
2. **anchor transport 생략** (`mode='no_transport'`) — $SO(3)\times SO(3)$로 붕괴
3. **$K + \varepsilon I$ regularization** — $\varepsilon I$는 congruence를 깨뜨림


In [15]:
torch.manual_seed(SEED)
neg_bracket = ModelPC2KNaiveBracket(WrenchPlueckerEncoder(k=8)).eval()
enc_no_tr = WrenchLearnableLiftEncoder(out_channels=8, k=8, mode='no_transport')

def neg1(R, p):
    A = coadjoint(R, p)
    with torch.no_grad():
        _, K = neg_bracket(P)
        _, KT = neg_bracket(transform_cloud(P, R, p))
    return scaled_err(KT, A @ K @ A.transpose(-1, -2))

def neg2(R, p):
    TP = transform_cloud(P, R, p)
    return scaled_err(enc_no_tr(TP), coad_apply(coadjoint(R, p), enc_no_tr(P)))

def neg3(R, p, eps=1e-3):
    A = coadjoint(R, p)
    with torch.no_grad():
        _, K = model_p(P)
        _, KT = model_p(transform_cloud(P, R, p))
    I = torch.eye(6)
    return scaled_err(KT + eps * I, A @ (K + eps * I) @ A.transpose(-1, -2))

rows_neg = [
    ('1. twist bracket on wrench', sweep(neg1)),
    ('2. no anchor transport    ', sweep(neg2)),
    ('3. K + eps*I  (eps=1e-3)  ', sweep(neg3)),
]
report(rows_neg, '[5] Negative controls  (p=0 통과, p!=0 에서 O(1) 실패가 정상)')


[5] Negative controls  (p=0 통과, p!=0 에서 O(1) 실패가 정상)
--------------------------------------------------------------------
                            p=0        |p|~1      |p|~1e2    |p|~1e4
1. twist bracket on wrench  7.4e-16  5.6e-02  1.0e+00  1.0e+00
2. no anchor transport      3.4e-15  9.6e-01  1.0e+00  1.0e+00
3. K + eps*I  (eps=1e-3)    3.7e-16  6.0e-01  8.9e-01  8.4e-01


## 6. 종합 판정


In [16]:
TOL_POS = 1e-9          # float64 구조적 성립 판정 (|p|~1e4 round-off까지 허용)
TOL_NEG = 1e-2          # negative control이 이보다 크게 실패해야 함 (p != 0 구간)

all_pos = rows_enc + rows_e2e + [('cascade', sweep(cascade_err))]
pos_ok = all(max(errs) < TOL_POS for _, errs in all_pos)
neg_ok = all(max(errs[1:]) > TOL_NEG for _, errs in rows_neg)

print(f'positive tests  (모든 법칙 성립, < {TOL_POS:.0e})      :', 'PASS' if pos_ok else 'FAIL')
print(f'negative controls (p!=0 에서 위반 감지, > {TOL_NEG:.0e}):', 'PASS' if neg_ok else 'FAIL')
assert pos_ok and neg_ok
print('\n=> 입력에 T를 가하면 출력이 정확히  L -> Ad_T^{-T} L,  K -> Ad_T^{-T} K Ad_T^{-1}  를 따른다.')

positive tests  (모든 법칙 성립, < 1e-09)      : PASS
negative controls (p!=0 에서 위반 감지, > 1e-02): PASS

=> 입력에 T를 가하면 출력이 정확히  L -> Ad_T^{-T} L,  K -> Ad_T^{-T} K Ad_T^{-1}  를 따른다.
